# Compilador de Faltas (versao Google Colab)

Versao adaptada do app de desktop (Tkinter) para rodar no Colab. O fluxo e:

1. Enviar a lista de turma (opcional) — `.csv`, `.xlsx` ou `.json`.
2. Enviar os arquivos de log de permanencia (`.txt`/`.csv`) — pode selecionar varios arquivos de uma vez, ou enviar um `.zip` com a pasta inteira.
3. Ajustar os parametros (limiar de presenca, turma padrao, filtro de datas).
4. Executar o processamento.
5. Baixar os relatorios gerados (`.xlsx` e `.csv`).

Rode as celulas em ordem, de cima para baixo.

## 0. Instalar dependencias

In [ ]:
!pip install -q openpyxl pandas


## Nucleo do compilador

Funcoes e classe `CompiladorFaltasCore` — identicas a versao desktop, sem a parte de interface grafica (Tkinter nao funciona no Colab).

In [ ]:
# -*- coding: utf-8 -*-
import os, re, csv, glob, json, logging, shutil, tempfile
from datetime import datetime, date
from collections import defaultdict, OrderedDict

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

try:
    import pandas as pd
except Exception:
    pd = None

LOG_FILENAME = "compilador_faltas.log"

# ==================== util ====================
def matricula_to_number(matricula):
    """Converte matrícula para número, removendo prefixos não numéricos"""
    if not matricula:
        return ""
    numeros = ''.join(filter(str.isdigit, str(matricula)))
    if numeros:
        return int(numeros)
    return matricula

def normalize_matricula(matricula):
    """Retorna apenas os dígitos da matrícula, para correspondência tolerante a prefixos (ex.: 'ES')."""
    if not matricula:
        return ""
    return ''.join(filter(str.isdigit, str(matricula)))

_NOMES_INVALIDOS = {"", "null", "none", "nan", "n/a", "-"}

def nome_valido(nome) -> bool:
    """Indica se um valor de nome é utilizável (não vazio e não um placeholder tipo 'null')."""
    if nome is None:
        return False
    return str(nome).strip().lower() not in _NOMES_INVALIDOS

def melhor_nome(atual: str, novo: str) -> str:
    """Escolhe o nome mais completo entre dois candidatos da mesma matrícula, ignorando
    valores inválidos (vazio, 'null', etc.) para que a matrícula seja sempre a identidade
    principal do aluno, mesmo quando o log traz ora o nome completo, ora só o primeiro nome."""
    if not nome_valido(novo):
        return atual
    if not nome_valido(atual):
        return novo
    return novo if len(str(novo).strip()) > len(str(atual).strip()) else atual

def setup_logging(saida_dir: str):
    try:
        os.makedirs(saida_dir, exist_ok=True)
    except Exception:
        saida_dir = os.getcwd()
    log_path = os.path.join(saida_dir, LOG_FILENAME)
    logging.basicConfig(
        level=logging.DEBUG,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_path, encoding="utf-8"),
            logging.StreamHandler()
        ],
        force=True
    )
    logging.info("Log inicializado em: %s", log_path)

def guess_delimiter(sample: str) -> str:
    import csv as _csv
    try:
        dialect = _csv.Sniffer().sniff(sample, delimiters=',;\t|')
        return dialect.delimiter
    except Exception:
        return ';' if sample.count(';') > sample.count(',') else ','

def read_text_or_csv(path: str):
    encodings = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252']
    import csv as _csv
    for enc in encodings:
        try:
            with open(path, 'r', encoding=enc, newline='') as f:
                sample = f.read(4096); f.seek(0)
                delim = guess_delimiter(sample)
                reader = _csv.DictReader(f, delimiter=delim)
                if reader.fieldnames:
                    reader.fieldnames = [fn.strip().replace('\ufeff', '') for fn in reader.fieldnames]
                for row in reader:
                    clean = {}
                    for k, v in row.items():
                        if k is None: continue
                        clean[k.strip().replace('\ufeff','')] = v
                    yield clean
            logging.info("Arquivo %s lido com codificação %s e delimitador '%s'", os.path.basename(path), enc, delim)
            return
        except UnicodeDecodeError:
            continue
        except Exception as e:
            logging.warning("Falha ao ler %s com %s: %s", os.path.basename(path), enc, e)
            continue
    logging.error("Não foi possível ler o arquivo: %s", path)

def format_header(ws, row_idx: int):
    font = Font(bold=True, color="FFFFFF")
    fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
    align = Alignment(horizontal="center", vertical="center")
    for col in range(1, ws.max_column + 1):
        cell = ws.cell(row=row_idx, column=col)
        cell.font = font
        cell.fill = fill
        cell.alignment = align

def autofit_columns(ws, min_w=12, max_w=50):
    for col in range(1, ws.max_column + 1):
        letter = get_column_letter(col)
        max_len = 0
        for row in range(1, ws.max_row + 1):
            v = ws.cell(row=row, column=col).value
            if v is None: continue
            max_len = max(max_len, len(str(v)))
        ws.column_dimensions[letter].width = max(min_w, min(max_len + 2, max_w))

def _wb_finalize_sheet(ws, header_row=1):
    """Finaliza a formatação da planilha - apenas congelamento de painéis"""
    try:
        ws.freeze_panes = ws.cell(row=header_row+1, column=1)
    except Exception as e:
        logging.warning("Erro ao congelar painel: %s", e)

def safe_save_workbook(wb, desired_path: str) -> str:
    if not desired_path.lower().endswith(".xlsx"):
        desired_path += ".xlsx"
    desired_dir = os.path.dirname(desired_path) or os.getcwd()
    try: os.makedirs(desired_dir, exist_ok=True)
    except Exception: pass
    base_name = os.path.basename(desired_path)
    with tempfile.TemporaryDirectory() as td:
        tmp_path = os.path.join(td, f"tmp_{base_name}")
        wb.save(tmp_path)
        try:
            if os.path.exists(desired_path):
                try: os.remove(desired_path)
                except Exception: pass
            shutil.move(tmp_path, desired_path)
            return desired_path
        except Exception as e:
            logging.warning("Falha/memória ao mover para %s: %s", desired_path, e)
        try:
            desktop = os.path.join(os.path.expanduser("~"), "Desktop")
            os.makedirs(desktop, exist_ok=True)
            stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            alt_path = os.path.join(desktop, base_name.replace(".xlsx", f"_{stamp}.xlsx"))
            shutil.move(tmp_path, alt_path)
            return alt_path
        except Exception:
            alt_path2 = os.path.join(os.getcwd(), base_name)
            shutil.move(tmp_path, alt_path2)
            return alt_path2

def safe_save_csv(rows, headers, desired_path: str) -> str:
    import csv as _csv
    if not desired_path.lower().endswith(".csv"):
        desired_path += ".csv"
    desired_dir = os.path.dirname(desired_path) or os.getcwd()
    try: os.makedirs(desired_dir, exist_ok=True)
    except Exception: pass
    base_name = os.path.basename(desired_path)
    with tempfile.TemporaryDirectory() as td:
        tmp_path = os.path.join(td, f"tmp_{base_name}")
        with open(tmp_path, 'w', encoding='utf-8-sig', newline='') as f:
            w = _csv.writer(f, delimiter=';')
            w.writerow(headers)
            for r in rows:
                w.writerow(r)
        try:
            if os.path.exists(desired_path):
                try: os.remove(desired_path)
                except Exception: pass
            shutil.move(tmp_path, desired_path)
            return desired_path
        except Exception as e:
            logging.warning("Falha/memória ao mover CSV para %s: %s", desired_path, e)
        try:
            desktop = os.path.join(os.path.expanduser("~"), "Desktop")
            os.makedirs(desktop, exist_ok=True)
            stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            alt_path = os.path.join(desktop, base_name.replace(".csv", f"_{stamp}.csv"))
            shutil.move(tmp_path, alt_path)
            return alt_path
        except Exception:
            alt_path2 = os.path.join(os.getcwd(), base_name)
            shutil.move(tmp_path, alt_path2)
            return alt_path2

# datas
def parse_user_date(text: str):
    if not text: return None
    text = text.strip()
    for fmt in ("%d/%m/%y", "%d/%m/%Y"):
        try: return datetime.strptime(text, fmt).date()
        except Exception: continue
    return None

def try_parse_date_from_filename(filename: str, regex_date):
    name = os.path.basename(filename)
    base = os.path.splitext(name)[0]
    if regex_date:
        m = re.search(regex_date, base)
        if m:
            txt = m.group(1) if m.groups() else m.group(0)
            for fmt in ("%Y-%m-%d","%d-%m-%Y","%d_%m_%Y","%Y%m%d","%d%m%Y","%d-%m-%Y_%H-%M-%S","%Y-%m-%d_%H-%M-%S"):
                try: return datetime.strptime(txt, fmt).strftime("%Y-%m-%d")
                except Exception: continue
            return txt
    for pat, fmt in [
        (r"(20\d{2}-\d{2}-\d{2})", "%Y-%m-%d"),
        (r"(20\d{2}\d{2}\d{2})", "%Y%m%d"),
        (r"(\d{2}-\d{2}-20\d{2})", "%d-%m-%Y"),
        (r"(\d{2}_\d{2}_20\d{2})", "%d_%m_%Y"),
    ]:
        m = re.search(pat, base)
        if m:
            txt = m.group(1)
            try:
                dt = datetime.strptime(txt, fmt)
                return dt.strftime("%Y-%m-%d")
            except Exception:
                pass
    return base

def dataid_to_date(data_id: str):
    if not data_id: return None
    for fmt in ("%Y-%m-%d","%d-%m-%Y","%d/%m/%Y","%Y/%m/%d","%Y%m%d","%d%m%Y"):
        try: return datetime.strptime(data_id, fmt).date()
        except Exception: continue
    m = re.search(r"(20\d{2}-\d{2}-\d{2})", data_id)
    if m:
        try: return datetime.strptime(m.group(1), "%Y-%m-%d").date()
        except Exception: return None
    return None

def parse_date_to_week(data_id: str):
    if not data_id: return ("","","")
    dt = dataid_to_date(data_id)
    if dt is None: return ("","",data_id)
    iso_year, iso_week, _ = dt.isocalendar()
    etiqueta = f"{iso_year}-W{iso_week:02d}"
    return (iso_year, iso_week, etiqueta)

def _normalizar_chave(k) -> str:
    """Normaliza um nome de coluna para comparação tolerante a acentos, espaços e maiúsculas.
    Alguns exports (ex.: JSON) já chegam com acentos/espaços removidos do próprio cabeçalho
    (ex.: 'Número de Identificação' vira 'nmerodeidentificao'); ao remover acentos e espaços
    dos dois lados da comparação, os dois formatos passam a bater."""
    if k is None:
        return ""
    s = str(k).strip().lower()
    return ''.join(ch for ch in s if ch.isascii() and ch.isalnum())

def _achatar_lista_json(obj):
    """Achata uma estrutura JSON que pode vir como lista aninhada (lista de listas) em uma
    lista plana de dicts (um dict por aluno)."""
    out = []
    if isinstance(obj, dict):
        out.append(obj)
    elif isinstance(obj, list):
        for item in obj:
            out.extend(_achatar_lista_json(item))
    return out

def load_classlist(path: str):
    if path is None or not os.path.exists(path): return OrderedDict()
    ext = os.path.splitext(path)[1].lower(); dados = []
    try:
        logging.info("Carregando lista de turma: %s", path)
        if ext in (".csv",".txt"):
            with open(path, 'r', encoding='utf-8-sig', newline='') as f:
                sample = f.read(4096); f.seek(0)
                delim = guess_delimiter(sample)
                logging.info("Detectado delimitador: '%s'", delim)
                rdr = csv.DictReader(f, delimiter=delim); dados = list(rdr)
        elif ext in (".xlsx",".xls"):
            if pd is None: raise RuntimeError("Pandas não disponível para ler Excel.")
            df = pd.read_excel(path, dtype=str); dados = df.to_dict(orient="records")
            logging.info("Arquivo Excel lido com %d linhas", len(dados))
        elif ext == ".json":
            with open(path, 'r', encoding='utf-8-sig') as f:
                bruto = json.load(f)
            dados = _achatar_lista_json(bruto)
            logging.info("Arquivo JSON lido com %d linhas", len(dados))
        else:
            raise RuntimeError("Formato de lista não suportado: %s" % ext)

        logging.info("Primeiras 3 linhas do arquivo:")
        for i, row in enumerate(dados[:3]):
            logging.info("  Linha %d: %s", i+1, dict(row))

        def _valor_ou_vazio(v):
            if v is None:
                return ""
            if pd is not None:
                try:
                    if pd.isna(v):
                        return ""
                except (TypeError, ValueError):
                    pass
            return v

        out = OrderedDict()
        for row in dados:
            keys = {_normalizar_chave(k): _valor_ou_vazio(row[k]) for k in row.keys() if k}
            mat = None
            for k in ("matrícula","matricula","id","registro","número de identificação","numero de identificacao"):
                kn = _normalizar_chave(k)
                if kn in keys and keys[kn]:
                    mat = str(keys[kn]).strip()
                    logging.info("Matrícula encontrada: %s (coluna: %s)", mat, k)
                    break

            nome = ""
            nome_parts = []
            if keys.get(_normalizar_chave('nome')):
                nome_parts.append(str(keys[_normalizar_chave('nome')]).strip())
            if keys.get(_normalizar_chave('sobrenome')):
                nome_parts.append(str(keys[_normalizar_chave('sobrenome')]).strip())
            if nome_parts:
                nome = ' '.join(nome_parts)
            if not nome:
                for k in ("nome","aluno","estudante"):
                    kn = _normalizar_chave(k)
                    if kn in keys and keys[kn]:
                        nome = str(keys[kn]).strip()
                        break

            turma = ""
            for k in ("turma","classe","disciplina","curso","grupos","grupo"):
                kn = _normalizar_chave(k)
                if kn in keys and keys[kn]:
                    turma = str(keys[kn]).strip()
                    logging.info("Turma encontrada: %s (coluna: %s) para %s", turma, k, mat)
                    break

            if mat:
                mat_norm = normalize_matricula(mat) or mat
                out[mat_norm] = {
                    'nome': nome,
                    'turma': turma
                }
        logging.info("Lista de turma carregada: %d alunos", len(out)); return out
    except Exception as e:
        logging.error("Erro ao ler lista de turma '%s': %s", path, e); return OrderedDict()

# núcleo
class CompiladorFaltasCore:
    def __init__(self, pasta_arquivos, limiar_min=45.0,
                 regex_data=None, lista_turma_path=None,
                 turma_padrao="", regex_turma=None,
                 inicio=None, fim=None):
        self.pasta_arquivos = pasta_arquivos
        self.limiar = float(limiar_min)
        self.regex_data = regex_data
        self.lista_turma = load_classlist(lista_turma_path) if lista_turma_path else OrderedDict()
        self.turma_padrao = turma_padrao
        self.regex_turma = regex_turma
        self.inicio = inicio
        self.fim = fim

        self.dados_presenca = defaultdict(list)
        self.dados_permanencia = []
        self.datas_processadas = set()
        self.semanas_processadas = set()

    def _dados_roster(self, matricula: str):
        """Retorna (nome, turma) da lista de turma para a matrícula (já normalizada, só dígitos)."""
        if not self.lista_turma:
            return "", ""
        dados = self.lista_turma.get(matricula)
        if isinstance(dados, dict):
            return dados.get('nome', ''), dados.get('turma', '')
        if dados:
            return str(dados), ""
        return "", ""

    def _dentro_intervalo(self, data_id: str) -> bool:
        if self.inicio is None and self.fim is None:
            return True
        dt = dataid_to_date(data_id)
        if dt is None:
            logging.warning("Não foi possível inferir data para filtrar: '%s' (incluído mesmo assim).", data_id)
            return True
        if self.inicio and dt < self.inicio:
            return False
        if self.fim and dt > self.fim:
            return False
        return True

    def ler_arquivos(self):
        arquivos = sorted(glob.glob(os.path.join(self.pasta_arquivos, "*.txt")) +
                          glob.glob(os.path.join(self.pasta_arquivos, "*.csv")))
        if not arquivos:
            logging.warning("Nenhum arquivo .txt/.csv encontrado em: %s", self.pasta_arquivos); return
        logging.info("Encontrados %d arquivos para processar.", len(arquivos))
        for arq in arquivos:
            self._processar_arquivo(arq)

    def _processar_arquivo(self, path: str):
        nome = os.path.basename(path)
        data_id = try_parse_date_from_filename(nome, self.regex_data)
        if not self._dentro_intervalo(data_id):
            logging.info("Ignorando por fora do intervalo: %s", nome)
            return
        turma_from_name = self._try_parse_turma_from_filename(nome)
        for row in read_text_or_csv(path):
            if row is None: continue
            self._processar_linha(row, data_id, turma_from_name)

    def _try_parse_turma_from_filename(self, filename: str) -> str:
        base = os.path.splitext(os.path.basename(filename))[0]
        if self.regex_turma:
            m = re.search(self.regex_turma, base)
            if m: return m.group(1) if m.groups() else m.group(0)
        return ""

    def _processar_linha(self, linha: dict, data_id: str, turma_from_name: str):
        try:
            matricula_bruta = None
            for key in ('Matrícula','Matricula','matrícula','matricula','ID','id','Registro','registro'):
                if key in linha and linha[key]: matricula_bruta = str(linha[key]).strip(); break
            if not matricula_bruta: return
            matricula = normalize_matricula(matricula_bruta) or matricula_bruta

            nome = ""
            for key in ('Nome','nome','Aluno','aluno','Estudante','estudante'):
                if key in linha and linha[key]: nome = str(linha[key]).strip(); break

            turma = ""
            for key in ('Turma','turma','Classe','classe','Disciplina','disciplina','Grupos','grupos','Grupo','grupo'):
                if key in linha and linha[key]: turma = str(linha[key]).strip(); break

            nome_roster, turma_roster = self._dados_roster(matricula)
            if nome_roster:
                nome = nome_roster
            if not turma:
                turma = turma_roster

            if not turma:
                turma = turma_from_name or self.turma_padrao or ""

            permanencia_str = None
            for key in ('Permanencia','Permanência','permanencia','permanência','Tempo','tempo','Permanência (min)','Permanencia (min)'):
                if key in linha:
                    permanencia_str = linha[key]
                    break
            if permanencia_str is None or str(permanencia_str).strip() == "": return
            try:
                permanencia = float(str(permanencia_str).replace(',', '.'))
            except Exception:
                m = re.search(r"[\d,.]+", str(permanencia_str)); permanencia = float(m.group(0).replace(',', '.')) if m else 0.0

            entrada = ""
            for key in ('Entrada','entrada','Hora','hora','CheckIn','checkin','Início','inicio','Inicio'):
                if key in linha and linha[key]: entrada = str(linha[key]).strip(); break

            saida = ""
            for key in ('Saída','saida','Saida','CheckOut','checkout','Fim','fim','Término','termino','Termino'):
                if key in linha and linha[key]: saida = str(linha[key]).strip(); break

            ano_iso, semana_iso, etiqueta = parse_date_to_week(data_id)
            if data_id: self.datas_processadas.add(data_id)
            if etiqueta: self.semanas_processadas.add(etiqueta)

            if permanencia > 0:
                rec = {
                    'matricula': matricula,
                    'nome': nome,
                    'data': data_id,
                    'entrada': entrada,
                    'saida': saida,
                    'tempo_permanencia': permanencia,
                    'turma': turma,
                    'ano_iso': ano_iso,
                    'semana_iso': semana_iso,
                    'etiqueta_semana': etiqueta
                }
                self.dados_permanencia.append(rec)
                if permanencia >= self.limiar:
                    presenca_rec = {'data': data_id,'nome': nome,'turma': turma,
                                   'tempo_permanencia': permanencia,'presente': True}
                    self.dados_presenca[matricula].append(presenca_rec)
        except Exception as e:
            logging.warning("Erro ao processar linha: %s | Erro: %s", linha, e)

    def _gerar_resumo_presencas(self, wb):
        ws = wb.create_sheet("Resumo de Presenças")
        ws.append([
            "Matrícula",
            "Nome",
            "Turma",
            "Número de Presenças (semanas, ≥ {} min no período)".format(self.limiar),
            "Semanas sem presença (0 dias)"
        ])
        todas_as_semanas = set(self.semanas_processadas)
        todas_matriculas = sorted(set(self.dados_presenca.keys()) | set(self.lista_turma.keys()))
        for matricula in todas_matriculas:
            presencas = self.dados_presenca.get(matricula, [])
            nome = ""
            for p in presencas:
                nome = melhor_nome(nome, p.get('nome',''))
            turma = presencas[0].get('turma',"") if presencas else ""
            semanas_com_presenca = set()
            for p in presencas:
                etiqueta_semana = parse_date_to_week(p.get('data', ''))[2]
                if etiqueta_semana:
                    semanas_com_presenca.add(etiqueta_semana)
            num = len(semanas_com_presenca)
            semanas_sem_presenca = max(len(todas_as_semanas) - len(semanas_com_presenca), 0)

            if not nome or not turma:
                nome_roster, turma_roster = self._dados_roster(matricula)
                if not nome: nome = nome_roster
                if not turma: turma = turma_roster

            ws.append([matricula_to_number(matricula), nome, turma, num, semanas_sem_presenca])
        format_header(ws, 1); autofit_columns(ws); _wb_finalize_sheet(ws, header_row=1); return ws

    def _gerar_detalhamento(self, wb):
        ws = wb.create_sheet("Detalhamento de Permanência")
        ws.append(["Matrícula","Nome","Turma","Data","Ano ISO","Semana ISO","Semana (ISO-YYYY-Www)","Entrada","Saída","Tempo de Permanência (min)"])
        for reg in sorted(self.dados_permanencia, key=lambda x: (x['turma'], x['matricula'], str(x['data']))):
            ws.append([matricula_to_number(reg['matricula']), reg['nome'], reg.get('turma',''), reg['data'], reg.get('ano_iso',''), reg.get('semana_iso',''),
                       reg.get('etiqueta_semana',''), reg.get('entrada',''), reg.get('saida',''), round(reg['tempo_permanencia'],2)])
        format_header(ws, 1)
        for col in (4,5,6,7,8,10):
            for r in range(2, ws.max_row+1): ws.cell(row=r, column=col).alignment = Alignment(horizontal="center")
        autofit_columns(ws); _wb_finalize_sheet(ws, header_row=1); return ws

    def _gerar_mapa_presencas(self, wb):
        datas = sorted(self.datas_processadas)
        alunos = sorted(set(list(self.dados_presenca.keys()) +
                             [d['matricula'] for d in self.dados_permanencia] +
                             list(self.lista_turma.keys())))

        pres_set = set(); nome_por_mat = {}; turma_por_mat = {}

        for mat, entradas in self.dados_presenca.items():
            for e in entradas:
                pres_set.add((mat, e['data']))
                if e.get('turma'):
                    turma_por_mat[mat] = e['turma']
                if e.get('nome'):
                    nome_por_mat[mat] = melhor_nome(nome_por_mat.get(mat, ''), e['nome'])
        for r in self.dados_permanencia:
            if r.get('nome'):
                nome_por_mat[r['matricula']] = melhor_nome(nome_por_mat.get(r['matricula'], ''), r['nome'])
            if r.get('turma') and r['matricula'] not in turma_por_mat:
                turma_por_mat[r['matricula']] = r['turma']

        for mat in alunos:
            if not nome_por_mat.get(mat) or not turma_por_mat.get(mat):
                nome_roster, turma_roster = self._dados_roster(mat)
                if not nome_por_mat.get(mat) and nome_roster:
                    nome_por_mat[mat] = nome_roster
                if not turma_por_mat.get(mat) and turma_roster:
                    turma_por_mat[mat] = turma_roster

        ws = wb.create_sheet("Mapa de Presenças")
        header = ["Matrícula","Nome","Turma"] + datas + ["Total Presenças","Total Faltas"]; ws.append(header)
        data_para_semana = {}
        for d in datas:
            data_para_semana[d] = parse_date_to_week(d)[2]
        total_semanas_processadas = len(set([s for s in data_para_semana.values() if s]))
        for mat in alunos:
            nome = nome_por_mat.get(mat, ""); turma = turma_por_mat.get(mat, "")
            linha = [matricula_to_number(mat), nome, turma]
            semanas_com_presenca = set()
            for d in datas:
                presenca = 'P' if (mat, d) in pres_set else 'F'
                if presenca == 'P':
                    etiqueta_semana = data_para_semana.get(d, '')
                    if etiqueta_semana:
                        semanas_com_presenca.add(etiqueta_semana)
                linha.append(presenca)
            pres_count = len(semanas_com_presenca)
            faltas = max(total_semanas_processadas - pres_count, 0)
            linha += [pres_count, faltas]
            ws.append(linha)
        format_header(ws, 1)
        for col_idx in range(4, 4 + len(datas) + 2):
            for row_idx in range(2, ws.max_row + 1): ws.cell(row=row_idx, column=col_idx).alignment = Alignment(horizontal="center")
        autofit_columns(ws); _wb_finalize_sheet(ws, header_row=1); return ws

    def _gerar_datas(self, wb):
        ws = wb.create_sheet("Datas Processadas"); ws.append(["#","Identificador de Data/Arquivo"])
        for i, d in enumerate(sorted(self.datas_processadas), start=1): ws.append([i, d])
        format_header(ws, 1); autofit_columns(ws); _wb_finalize_sheet(ws, header_row=1); return ws

    def _gerar_pivot_semanal(self, wb):
        semanas = sorted([r['etiqueta_semana'] for r in self.dados_permanencia if r.get('etiqueta_semana')])
        semanas = sorted(list(set(semanas)))
        soma = defaultdict(lambda: defaultdict(float)); nomes = {}
        for r in self.dados_permanencia:
            sem = r.get('etiqueta_semana','')
            if not sem: continue
            key = (r.get('turma',''), r['matricula'])
            soma[key][sem] += float(r.get('tempo_permanencia', 0.0))
            nomes[key] = melhor_nome(nomes.get(key, ''), r.get('nome',''))
        ws = wb.create_sheet("Permanência por Semana")
        header = ["Turma","Matrícula","Nome"] + semanas + ["Total (min)"]; ws.append(header)
        for key in sorted(soma.keys(), key=lambda x: (x[0], x[1])):
            turma, mat = key; nome = nomes.get(key, ''); linha = [turma, matricula_to_number(mat), nome]; total = 0.0
            for s in semanas:
                val = round(soma[key].get(s, 0.0), 2); total += val; linha.append(val)
            linha.append(round(total,2)); ws.append(linha)
        format_header(ws, 1)
        for col_idx in range(4, 4 + len(semanas) + 1):
            for row_idx in range(2, ws.max_row + 1): ws.cell(row=row_idx, column=col_idx).alignment = Alignment(horizontal="center")
        autofit_columns(ws); _wb_finalize_sheet(ws, header_row=1); return ws

    def export_csv_concatenado(self, desired_csv_path):
        headers = ["Matrícula","Nome","Turma","Data","Semana ISO","Entrada","Saída","Tempo de Permanência (min)"]
        rows = []
        for reg in sorted(self.dados_permanencia, key=lambda x: (x['turma'], x['matricula'], str(x['data']))):
            rows.append([
                matricula_to_number(reg['matricula']),
                reg.get('nome',''),
                reg.get('turma',''),
                reg.get('data',''),
                reg.get('semana_iso',''),
                reg.get('entrada',''),
                reg.get('saida',''),
                round(reg.get('tempo_permanencia',0.0),2)
            ])
        if desired_csv_path is None:
            desired_csv_path = os.path.join(self.pasta_arquivos, "relatorios_faltas_v5.csv")
        final_csv = safe_save_csv(rows, headers, desired_csv_path)
        logging.info("CSV salvo em: %s", final_csv)
        return final_csv

    def gerar_relatorios(self, arquivo_saida_xlsx=None, arquivo_saida_csv=None):
        if arquivo_saida_xlsx is None:
            arquivo_saida_xlsx = os.path.join(self.pasta_arquivos, "relatorios_faltas_v5.xlsx")

        turmas_encontradas = any(r.get('turma') for r in self.dados_permanencia)
        if not turmas_encontradas and not self.lista_turma:
            logging.warning("ATENÇÃO: Nenhuma informação de turma encontrada! Carregue uma lista de turma, defina uma turma padrão, ou use Regex Turma.")
        elif not turmas_encontradas and self.lista_turma:
            logging.warning("ATENÇÃO: Lista de turma carregada, mas as matrículas não correspondem aos logs.")

        csv_final = self.export_csv_concatenado(arquivo_saida_csv)
        logging.info("Gerando XLSX: %s", arquivo_saida_xlsx)
        wb = Workbook()
        std = wb.active
        if std is not None:
            wb.remove(std)
        self._gerar_resumo_presencas(wb)
        self._gerar_detalhamento(wb)
        self._gerar_mapa_presencas(wb)
        self._gerar_datas(wb)
        self._gerar_pivot_semanal(wb)
        xlsx_final = safe_save_workbook(wb, arquivo_saida_xlsx)
        logging.info("Excel salvo em: %s", xlsx_final)
        return xlsx_final, csv_final

    def processar(self, arquivo_saida_xlsx=None, arquivo_saida_csv=None):
        logging.info("=== COMPILADOR DE FALTAS (Colab) ===")
        self.ler_arquivos()
        logging.info("Datas: %d | Semanas: %d | Registros permanência: %d",
                     len(self.datas_processadas), len(self.semanas_processadas), len(self.dados_permanencia))
        return self.gerar_relatorios(arquivo_saida_xlsx, arquivo_saida_csv)

print("Núcleo carregado.")


## 1. Upload da lista de turma (opcional)

Aceita `.csv`, `.xlsx` ou `.json`. Se voce nao tiver uma lista, apenas rode a celula e clique em **Cancel** na janela de upload — o processamento continua sem ela (os nomes/turmas virao so dos proprios arquivos de log, quando disponiveis).

In [ ]:
from google.colab import files
import os

os.makedirs("/content/lista_turma", exist_ok=True)
lista_turma_path = None

print("Selecione o arquivo da lista de turma (ou cancele para pular)...")
uploaded = files.upload()
for nome_arquivo, conteudo in uploaded.items():
    destino = os.path.join("/content/lista_turma", nome_arquivo)
    with open(destino, "wb") as f:
        f.write(conteudo)
    lista_turma_path = destino

print("Lista de turma:", lista_turma_path or "(nenhuma enviada)")


## 2. Upload dos arquivos de log (.txt/.csv)

Selecione varios arquivos de uma vez no dialogo de upload, **ou** compacte a pasta inteira em um `.zip` e envie o zip — ele e extraido automaticamente.

In [ ]:
from google.colab import files
import os, shutil, zipfile

pasta_logs = "/content/logs"
if os.path.exists(pasta_logs):
    shutil.rmtree(pasta_logs)
os.makedirs(pasta_logs, exist_ok=True)

print("Selecione os arquivos de log (.txt/.csv) ou um .zip com a pasta inteira...")
uploaded = files.upload()
for nome_arquivo, conteudo in uploaded.items():
    destino = os.path.join(pasta_logs, nome_arquivo)
    with open(destino, "wb") as f:
        f.write(conteudo)
    if nome_arquivo.lower().endswith(".zip"):
        with zipfile.ZipFile(destino) as zf:
            zf.extractall(pasta_logs)
        os.remove(destino)

arquivos_encontrados = [f for f in os.listdir(pasta_logs) if f.lower().endswith((".txt", ".csv"))]
print(f"{len(arquivos_encontrados)} arquivo(s) .txt/.csv prontos em {pasta_logs}")


## 3. Parâmetros

Preencha o formulário abaixo (não precisa editar código) e rode a célula.

In [ ]:
#@markdown Preencha os campos e rode esta célula (não precisa editar código).

limiar_min = 45 #@param {type:"number"}
turma_padrao = "" #@param {type:"string"}
nome_arquivo_saida = "relatório_faltas_estudo dirigido" #@param {type:"string"}

#@markdown Datas no formato **dd/mm/aaaa** (ou dd/mm/aa). Deixe em branco para não filtrar.
data_inicio_txt = "" #@param {type:"string"}
data_fim_txt = "" #@param {type:"string"}

#@markdown Avançado — regex opcionais para extrair data/turma do *nome do arquivo* (deixe em branco se não usar):
regex_data = "" #@param {type:"string"}
regex_turma = "" #@param {type:"string"}

regex_data = regex_data or None
regex_turma = regex_turma or None
inicio = parse_user_date(data_inicio_txt) if data_inicio_txt else None
fim = parse_user_date(data_fim_txt) if data_fim_txt else None

print("Limiar:", limiar_min, "min | Turma padrão:", turma_padrao or "(nenhuma)")
print("Nome dos arquivos de saída:", nome_arquivo_saida)
print("Início:", inicio, "| Fim:", fim)


## 4. Executar processamento

In [ ]:
setup_logging("/content")

comp = CompiladorFaltasCore(
    pasta_arquivos=pasta_logs,
    limiar_min=limiar_min,
    regex_data=regex_data,
    lista_turma_path=lista_turma_path,
    turma_padrao=turma_padrao,
    regex_turma=regex_turma,
    inicio=inicio,
    fim=fim,
)

xlsx_path, csv_path = comp.processar(
    arquivo_saida_xlsx=f"/content/{nome_arquivo_saida}.xlsx",
    arquivo_saida_csv=f"/content/{nome_arquivo_saida}.csv",
)
print("XLSX:", xlsx_path)
print("CSV:", csv_path)


## 5. Baixar os relatórios para o seu PC

O Colab roda na nuvem, então não grava direto numa pasta do seu computador. Ao rodar a célula abaixo, o navegador abre o download de cada arquivo (normalmente vão para a pasta **Downloads** do seu PC), já com o nome que você escolheu no formulário do passo 3.

In [ ]:
from google.colab import files

print("Baixando para o seu PC (verifique a pasta Downloads do navegador)...")
files.download(xlsx_path)
files.download(csv_path)
print("Pronto:", xlsx_path.split("/")[-1], "e", csv_path.split("/")[-1])
